# IBM November 2025 Challenge

https://research.ibm.com/haifa/ponderthis/challenges/November2025.html

## 0. Introduction

We set $n = 10^{100}$ and $N = n + 1000$, and let $f$ be the function used to construct a new word (different in each case).

We are interested in the characters in positions $n, \dots, N - 1$ for the words  
$$
f^n(\text{CAT}) \text{ and } f^n(\text{RABBITS}).
$$

In [1]:
n = 10**100
N = n + 1000

# In both functions below, it is assumed that `lengths` has already been computed

def compute_min_lengths_iters():
    ### Compute lengths of f^k(c) for each character c and iteration k ###
    lengths = {}

    # Base case: at iteration 0, each character has length 1
    for c in letters:
        lengths[(0,c)] = 1

    k=0
    # Keep iterating until for every letter, f^k(c) has length >= N
    while any(lengths[(k,c)]<N for c in letters):
        k+=1
        for c in letters:
            # Length of f^k(c) is the sum of lengths of f^(k-1)(d) for each d in letters[c]
            lengths[(k,c)] = sum( lengths[(k-1,d)] for d in letters[c] )
    ###
    return lengths,k



def find(s0,k,pos,lengths=None):
    # If 'lengths' is not provided, compute a dictionary storing
    # the lengths of the words f^i(c) for each character c and iteration i.
    if lengths==None:
        lengths = {}

        # Base case: for iteration 0, each character has length 1
        for c in letters:
            lengths[(0,c)] = 1

        # For each iteration i, compute the total length of f^i(c)
        # by summing the lengths of f^(i-1)(d) for all d in letters[c].
        for i in range(1,k+1):
            for c in letters:
                lengths[(i,c)] = sum( lengths[(i-1,d)] for d in letters[c] )

    def search(s0,k,pos):
        # Base case: at level 0, we directly return the character at position 'pos'
        if k==0:
            return s0[pos]

        #case k>0
        # Recursive case: determine in which part of f^k(s0) the position 'pos' lies
        l = 0
        for c in s0:
            l += lengths[(k,c)]
            if l>=pos+1: #+1 because 'pos' is 0-based
                break

        # Recurse into the appropriate sub-block:
        # * letters[c] is the expansion of character c
        # * k-1 because we go one level deeper
        # * pos adjusted by subtracting the lengths of the previous parts
        return search(letters[c],k-1,pos-(l-lengths[(k,c)]))

    # Start the recursive search
    return search(s0,k,pos)

### 'CAT'

1. We have that `len(f^{480}(c)) >= N` for any $c \in \{G, T, C, A\}$.

2. We have $n = 480q + 160$.

3. It is worth noting that $C \mapsto TG \mapsto CAT$, that is, after an even number of iterations of the letter $C$, the first letter is again $C$. Hence, $f^{160}(C)[0] = C = f^{480}(C)[0]$.

All in all, we have
$$
f^n(\text{CAT})[:N] = f^n(C)[:N] = f^{480q+160}(C)[:N] = f^{480q}(f^{160}(C))[:N]
= f^{480q}(C)[:N] = (\text{even iteration}) = f^{480(q-1)}(C)[:N] = \cdots = f^{480}(C)[:N].
$$

Therefore, we need to search for $f^{480}(C)[i]$ for $i = n, \dots, N-1$.


In [2]:
# Define the substitution rules for each letter
letters = {'G':'T',
           'T':'CA',
           'C':'TG',
           'A':'C'}

### Compute lengths of f^k(c) for each character c and iteration k ###
lengths,k = compute_min_lengths_iters()

print(f"{k} (pair) is the minimum number of iteratiors to have that each character is mapped to a work of length at least N=10**100+1000.")

r = n%480
print(f"The remained of n=10**100 divided by {k} is {r}, which is pair again.")

sol = ''
for i in range(n,N):
    sol += find('C',k,i,lengths)

print("The solution is:")
print(sol)

480 (pair) is the minimum number of iteratiors to have that each character is mapped to a work of length at least N=10**100+1000.
The remained of n=10**100 divided by 480 is 160, which is pair again.
The solution is:
TGTGCTGCCATGCCACATTGCCACATCATTGCATTGTGCTGCCATGCCACATTGCCACATCATTGTGCCACATCATTGCATTGTGCTGCCACATCATTGCATTGTGCCATTGTGCTGCCACATTGTGCTGCCATGCCACATCATTGTGCTGCCATGCCACATTGCCACATCATTGCATTGTGCTGCCATGCCACATTGCCACATCATTGTGCCACATCATTGCATTGTGCCATTGTGCTGCCATGCCACATTGCCACATCATTGTGCCACATCATTGCATTGTGCTGCCACATCATTGCATTGTGCCATTGTGCTGCCACATTGTGCTGCCATGCCACATTGCCACATCATTGTGCCACATCATTGCATTGTGCTGCCACATCATTGCATTGTGCCATTGTGCTGCCATGCCACATCATTGCATTGTGCCATTGTGCTGCCACATTGTGCTGCCATGCCACATTGCCACATCATTGCATTGTGCCATTGTGCTGCCACATTGTGCTGCCATGCCACATCATTGTGCTGCCATGCCACATTGCCACATCATTGTGCCACATCATTGCATTGTGCCATTGTGCTGCCACATTGTGCTGCCATGCCACATCATTGTGCTGCCATGCCACATTGCCACATCATTGCATTGTGCTGCCATGCCACATTGCCACATCATTGTGCCACATCATTGCATTGTGCTGCCACATCATTGCATTGTGCCATTGTGCTGCCACATTGTGCTGCCATGCCACATCATTGTGCTGCCATGCCACATTGCCACATCAT

### 'RABBITS'

1. We have that `len(f^{480}(c)) >= N` for any $c \in \{G, T, C, A, R, B, I, S\}$.

We note that there is no way to obtain 'R' again as the first letter after applying $f$ to any single character. Instead, we take one step to obtain 'B', which then enters the loop (considering only the first letter in each image)
$$
B \mapsto I \mapsto T \mapsto C \mapsto B.
$$

Thus, we reduce to
$$
f^n(\text{RABBITS})[:N] = f^n(R)[:N] = f^{n-1}(B)[:N].
$$

Now we proceed as before:

2. We have $n - 1 = 480q + 159$.

3. Since $B \mapsto I \mapsto T \mapsto C \mapsto B$ is a 4-cycle and $159 \equiv 3 \pmod{4}$, we have $f^{159}(B)[0] = C$.

All in all, we have
$$
f^n(\text{RABBITS})[:N] = f^n(R)[:N] = f^{n-1}(B)[:N] = f^{480q+159}(B)[:N]
= f^{480q}(f^{159}(B))[:N] = f^{480q}(C)[:N] = (\text{4-cycle and } 480 \equiv 0 \pmod{4})
= f^{480(q-1)}(C)[:N] = \cdots = f^{480}(C)[:N].
$$

Therefore, we have to search for $f^{480}(C)[i]$ for $i = n, \dots, N - 1$.


In [3]:
# Define the substitution rules for each letter
letters = {'G':'T',
           'T':'CA',
           'C':'BR',
           'A':'I',
           'R':'B',
           'B':'IS',
           'I':'TG',
           'S':'C'}

### Compute lengths of f^k(c) for each character c and iteration k ###
lengths,k = compute_min_lengths_iters()

print(f"{k} is the minimum number of iterations such that each character is mapped to a word of length at least N = 10**100 + 1000.")

r = (n - 1) % 480
print(f"The remainder of n - 1 = 10**100 - 1 divided by {k} is {r}.")

print(f"The remainder of {r} divided by 4 is {r % 4}, so f^{r}(B) = C.")
print(f"The remainder of {k} divided by 4 is {k % 8}, so f^{k}(C) = C.")

sol = ''
for i in range(n,N):
    sol += find('C',480,i,lengths)

print("The solution is:")
print(sol)

480 is the minimum number of iterations such that each character is mapped to a word of length at least N = 10**100 + 1000.
The remainder of n - 1 = 10**100 - 1 divided by 480 is 159.
The remainder of 159 divided by 4 is 3, so f^159(B) = C.
The remainder of 480 divided by 4 is 0, so f^480(C) = C.
The solution is:
BRTGCBRICATGCISCATBRICAISBCATBRISBTGBRITGCISBRICAISBTGCISCATISBTGBRICAISBCATBRISBTGBRITGCISCATISBTGCATBRTGCISBTGBRITGCISCATBRTGCBRICATGCISCATISBTGBRITGCISBRICAISBTGCISCATISBTGCATBRTGCBRICATGCISCATBRICAISBCATBRTGCISCATISBTGCATBRTGCISBTGBRITGCISBRICAISBTGCISCATISBTGBRICAISBCATBRISBTGBRITGCISCATISBTGCATBRTGCISBTGBRITGCISCATBRTGCBRICATGCISCATBRICAISBCATBRTGCISCATISBTGCATBRTGCBRICAISBCATBRISBTGBRICATBRTGCBRICATGCISCATISBTGCATBRTGCISBTGBRITGCISCATBRTGCBRICATGCISCATBRICAISBCATBRISBTGBRICATBRTGCBRICAISBTGBRITGCISBRICAISBCATBRTGCBRICATGCISCATBRICAISBCATBRTGCISCATISBTGCATBRTGCISBTGBRITGCISCATBRTGCBRICATGCISCATISBTGBRITGCISBRICAISBTGCISCATISBTGCATBRTGCBRICATGCISCATBRICAISBCATBRTGCISCATIS